In [ ]:
curl -sSL https://install.python-poetry.org | POETRY_HOME=/data/bma/envs/poetry python3 -
cd /data/bma/nicheflow && /data/bma/envs/poetry/bin/poetry install
source /data/bma/envs/poetry/venv/bin/activate

In [ ]:
# 基础依赖
pip install torch==2.5.1 \
    scanpy==1.11.4 \
    numpy==2.3.3 \
    tqdm==4.67.1 \
    torch-geometric==2.6.1 \
    scipy==1.16.2 \
    matplotlib==3.10.6 \
    flax==0.11.2 \
    diffrax==0.7.0 \
    jaxtyping==0.3.2 \
    lightning==2.5.5 \
    torchmetrics==1.8.2 \
    hydra-core==1.3.2 \
    hydra-submitit-launcher==1.2.0 \
    wandb==0.22.0 \
    torchcfm==1.0.7 \
    torchdyn==1.0.6 \
    tabulate==0.9.0 \
    omegaconf \
    ruff>=0.13.1 \
    mypy>=1.18.2 \
    ipykernel>=6.30.1

# 安装 moscot（需要从 git 安装）
pip install git+https://github.com/theislab/moscot.git

In [ ]:
wget -p /workspace/nicheflow/data https://doi.org/10.6084/m9.figshare.30426610/files/data

### train

In [ ]:
cd /data/bma/nicheflow && python -m nicheflow.train experiment=nicheflow/glvfm/med
# python nicheflow/train.py experiment=nicheflow/glvfm/med

In [ ]:
nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total --format=csv
watch -n 1 nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv


In [ ]:
Epoch 0:   1%| | 1000/100000 [03:30<5:47:49,  4.74it/s, v_num=im0l, train/loss=0.455, train/lo
Your vector field does not have `nn.Parameters` to optimize.              | 0/8 [00:00<?, ?it/s]
/workspace/env/nicheflow/lib/python3.11/site-packages/torchdyn/numerics/odeint.py:83: UserWarning: Setting tolerances has no effect on fixed-step methods
  warn("Setting tolerances has no effect on fixed-step methods")
/workspace/env/nicheflow/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 400. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 1/8 [00:01<00:08,  0.83it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 2/8 [00:01<00:04,  1.41it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 3/8 [00:01<00:02,  1.84it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 4/8 [00:01<00:01,  2.17it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 5/8 [00:02<00:01,  2.14it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.      | 6/8 [00:02<00:00,  2.13it/s]
                                                                                              Your vector field does not have `nn.Parameters` to optimize.█▋    | 7/8 [00:03<00:00,  2.12it/s]
Epoch 0:   1%|▏                | 1358/100000 [04:49<5:50:29,  4.69it/s, v_num=im0l, train/loss=0.455, train/loss_x=0.381, trEpoch 0:   1%| | 1368/100000 [04:51<5:50:30,  4.69it/s, v_num=im0l, train/loss=0.409, train/loss_x=0.336, train/loss_pos=0.0Epoch 0:   1%| | 1374/100000 [04:52<5:50:19,  4.69it/s, v_num=im0l, train/loss=0.464, train/loss_x=0.389, train/loss_pos=0.0Epoch 0:   1%| | 1374/100000 [04:52<5:50:26,  4.69it/s, v_num=im0l, train/loss=0.469, train/loss_x=0.393, train/loss_pos=0.0Epoch 0:   1%|▏         | 1463/100000 [05:11<5:49:25,  4.70it/s, v_num=im0l, train/loss=0.413, train/loss_x=0.337, train/loss_pos=0.0754]^C[12:09:23][pytorch_lightning.utilities.rank_zero][INFO] 
Detected KeyboardInterrupt, attempting graceful shutdown ...
wandb: 
wandb: You can sync this run to the cloud by running:
wandb: wandb sync /workspace/nicheflow/outputs/2026-02-28/12-03-17/wandb/offline-run-20260228_120404-gjgwim0l
wandb: Find logs at: outputs/2026-02-28/12-03-17/wandb/offline-run-20260228_120404-gjgwim0l/logs
Epoch 0:   1%|▏         | 1463/100000 [05:11<5:50:09,  4.69it/s, v_num=im0l, train/loss=0.413, train/loss_x=0.337, train/loss_pos=0.0754]

### eval

In [ ]:
python python /workspace/nicheflow/nicheflow/eval.py experiment=... ckpt_path=...

# NOTES

整个项目采用了 **PyTorch Lightning + Hydra** 的现代化标准深度学习框架进行构建。以下是该项目代码框架的详细介绍：

### 1. 代码整体执行 Pipeline (工作流)
项目的整个生命周期（从配置读取到模型评估）如下：
1. **统一配置读取：** 通过终端执行 `python nicheflow/train.py experiment=...`。此时，**Hydra** 会解析 `configs/` 文件夹下游的一系列 `.yaml` 配置文件组合（包含优化器、数据路径、模型定义等），动态注入所需参数。
2. **模块实例化：** `train.py` 作为程序入口点，会并行初始化三核组件：
   - **`LightningDataModule`** (比如 `microenv_datamodule.py`)。
   - **`LightningModule`** 即任务包装器 (比如 `tasks/flow_matching.py`)。
   - **`Trainer`** 负责将两者组合并驱动运行。
3. **数据预处理与加载：** 在 `LightningDataModule` 进行准备时，读取经预处理的 H5AD 数据集。数据加载器 `datasets/st_dataset_base.py` 会负责处理复杂的匹配（如通过 `torchcfm` 的 `OTPlanSampler` 使用最优传输计算匹配不同时间节点的细胞微环境配对）。
4. **流匹配与反向传播：** 训练阶段，微环境的点云会作为输入传入模型主干（如 `pc_transformer.py`）。通过预设的 `GLVFMLoss` 或 `GVFMLoss`（`losses.py`）计算连续流匹配的梯度，通过反向传播更新模型。
5. **采样与评估：** 评估阶段（`eval.py`），模型使用 `torchdyn.core.NeuralODE` 求解常微分方程提取生成流，获得目标时间节点的模拟点云，并可能经由预训练好的分类器（`ct_classification.py`）评估生成效果。

---

### 2. 核心文件夹及类文件作用

核心代码主要位于 nicheflow 目录下（注意第一个是根目录，第二个是实际上真正存放 Python Module 的包）：

#### 📍 训练与评测入口
* **`train.py`**：负责读取 Hydra 参数并调用 PyTorch Lightning 的 Trainer 执行 `fit()` 和 `test()` 函数完成训练、保存和测试逻辑。
* **`eval.py` 及相关衍生脚本** (`eval_state_dict_ckpts.py` 等)：使用独立检查点（`.ckpt`）进行生成结果测试及消融实验 (Ablations)。

#### 📍 `nicheflow/tasks/`（任务调度器）
这层由 PyTorch Lightning（`LightningModule`）构成，包装了模型、验证逻辑并定义了 `training_step()` 与 `validation_step()`：
* **`flow_matching.py`**：训练 NicheFlow/RPCFlow/SPFlow 所对应的流匹配任务，控制梯度与核心训练循环。
* **`ct_classification.py`**：训练用于对最终生成的微环境图谱进行度量的下游 Cell Type 评估分类器。

#### 📍 `nicheflow/models/`（模型架构层）
这里存放了核心的底层 `torch.nn.Module` 数学和架构结构：
* **`backbones/pc_transformer.py`**：点云 Transformer 主干网络（模型基于序列或图对微环境点云提取特征），是 NicheFlow / RPCFlow 的核心。
* **`backbones/spmlp.py`**：单点多层感知机（SinglePointMLP），是基线模型 SPFlow（Single Point Flow）的核心。
* **`flows.py`**：定义了构建微分方程生成轨迹的流模型体系架构（`PointCloudFlow` 和 `SinglePointFlow`）。
* **`losses.py`**：不同变分目标的损失函数定义：包含 `CFMLoss`（常规流）、`GVFMLoss`、`GLVFMLoss`（论文核心指标）。
* **`ct_classifier.py`**：细胞类型分类器的网络结构定义。

#### 📍 `nicheflow/datamodules/` 与 `nicheflow/datasets/`（数据层）
使用了 PyTorch Dataset 与 Lightning Datamodule 结合的架构：
* **`microenv_datamodule.py` 及 `microenv_dataset.py`**：用于处理真正的空间点云（微环境）任务格式。
* **`sp_and_rpc_datamodule.py` 及 `rpc_dataset.py`**：用于处理点处理（Single Point）和随机切分点云（Randomized Point Cloud）基线实验的数据。
* **`st_dataset_base.py`**：时序和空间数据特征处理基类，利用了最优传输（Optimal Transport - OT）来生成源域到目标域的 Pair 配对轨迹。
* **`h5ad_ct_dataset.py`**：专门用来训练 Cell Type 分类器的数据加载器，操作 AnnData (`.h5ad`) 文件。

#### 📍 其他支线文件夹
* **`nicheflow/preprocessing/`**：`h5ad_preprocessor.py` 等执行数据清洗，QC 质控并对空间转录图表做标准化抽象。
* **`nicheflow/transforms/`**：定义对 DataLoader 产出时的独立增广算子（如对坐标数据施加自定义 `one_hot_encode_slice.py` 变换）。
* **`nicheflow/utils/`**：辅助性操作，例如固定种子（`seed.py`）、日志辅助仪（`log.py`）、绘图函数（`plots.py`）。

---

### 3. 组件间的关联链路机制（以 NicheFlow 训练为例）

要快速理解整体关联，你可以把整个项目当作一台装配线来看：

1. **装配站初始化 (Hydra configs -> `train.py`):** 
   当你下达带有参数文件的执行命令（例如 `experiment=nicheflow/glvfm/med`）时，`train.py` 第一步就是把配置文件的需求转化为具体的类实例化。它分别呼叫 `microenv_datamodule.py` 生成传送带，和 `flow_matching.py`（任务）作为机床。
2. **建立基础模型结构模型 (`flow_matching.py` -> `models/`)**: 
   `flow_matching.py` 在被初始化时，它会向 `models/` 拿来 `flows.py` 中的组合包装，并且 `flows.py` 内部又加载了来自 `backbones/pc_transformer.py` 作为深度学习可调参数层网络架构，并将 `losses.py` 中的 `GLVFMLoss` 直接接入当作指导其方向的指标函数。
3. **数据流入与计算 (`microenv_datamodule.py` -> `dataset` -> Task):** 
   数据在 `datasets/microenv_dataset.py` 被包装成 Batch 并使用最优传输（OT）算出哪组微环境 $t_0$ 需要向哪个 $t_1$ 配对（生成配对点云）。随后，Batches 会自动被输入到 `tasks/flow_matching.py` 中的 `training_step()` 里。
4. **最终生成输出 (`training_step` -> ODE Solver):**
   模型将处理好的配对交给 `models.PointCloudFlow` 接受推导并给出流向量并计算 Loss 回传。在验证 / 预测输出阶段，网络实际上通过 `torchdyn.core.NeuralODE`（神经微分方程求解器）使用这个 Transformer 逐帧步进（积分）还原出时间段的微环境预测图像图谱，并交由评价组（Classifier）打分。

In [2]:
import pickle; data = pickle.load(open('/data/bma/nicheflow/data/embryonic_data.pkl', 'rb')); print(type(data));

ModuleNotFoundError: No module named 'torch'

- 多GPU：增加devices

---

```YAML
defaults:
  - default

accelerator: gpu
devices: [1, 3]     # 手动指定使用的具体的 GPU ID
# devices: "auto"   # 或-1。它会吃掉它能看见的所有可用卡

strategy: ddp
use_distributed_sampler: True
```
---

- 多CPU：增加num_workers # 设置为你CPU逻辑核心数的 1/4 到 1/2